# pyannote.audio 単体による話者分離（Diarization）検証ノートブック

このノートブックは `pyannote.audio` を単体で使用し、音声ファイルから高精度な話者分離とタイムスタンプ解析を行うためのものです。

## 1. 依存ライブラリのインストール
Kaggle 標準環境には `pyannote.audio` が入っていないため、インストールを行います。

In [ ]:
# SciPy を最新版に更新し、NumPy 2.x との非互換エラーを完全解消
!pip install -q --upgrade scipy pyannote-audio

## 2. 設定と Hugging Face アクセストークンの入力

In [ ]:
import os
from huggingface_hub import login

# 1. Hugging Face のアクセストークン(hf_...)を入力してください
HF_TOKEN = "Hugging Face トークン"

# 2. 解析対象の音声/動画ファイルのパス
AUDIO_FILE_PATH = "/kaggle/input/trimming/trimmed_videos/sfw.mp4"

# トークンでログイン処理
if HF_TOKEN and not HF_TOKEN.startswith("YOUR_"):
    login(token=HF_TOKEN)
    print("✅ Hugging Face へのログインが完了しました。")
else:
    print("⚠️ 注意: HF_TOKEN に正しいアクセストークン(hf_...)を設定してください。")

## 3. モデルのロードと GPU 設定

In [ ]:
import torch
from pyannote.audio import Pipeline

print("モデル (pyannote/speaker-diarization-3.1) をロード中...")

# 公式パイプラインのロード
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=HF_TOKEN
)

# GPU への転送
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline.to(device)

print(f"✅ ロード成功！ 使用デバイス: {device}")

## 4. 話者分離（Diarization）の実行

In [ ]:
import pandas as pd

print(f"話者分離を実行中... 対象ファイル: {AUDIO_FILE_PATH}")

# パイプラインの実行
diarization = pipeline(AUDIO_FILE_PATH)

# 結果を DataFrame 化
records = []
for turn, _, speaker in diarization.itertracks(yield_label=True):
    records.append({
        "start": round(turn.start, 2),
        "end": round(turn.end, 2),
        "duration": round(turn.duration, 2),
        "speaker": speaker
    })

df = pd.DataFrame(records)
print(f"✅ 解析完了！ 検出された発話数: {len(df)} 件")

## 5. 話者分析とサマリー表示

In [ ]:
# 話者ごとの発話合計時間サマリー
print("=== 話者ごとの発話合計時間サマリー ===")
summary = df.groupby("speaker")["duration"].agg(["count", "sum"]).reset_index()
summary.columns = ["話者ID", "発話件数", "合計時間(秒)"]
summary["合計時間(分)"] = (summary["合計時間(秒)"] / 60).round(2)
display(summary)

print("\n=== 全発話タイムスタンプ一覧（上位20件） ===")
display(df.head(20))

## 6. ターゲット話者の 5〜10秒 発話の切り出し保存

In [ ]:
import subprocess
from pathlib import Path

# 対象話者IDと抽出したい長さ範囲(秒)
TARGET_SPEAKER = "SPEAKER_00"
MIN_SEC = 5.0
MAX_SEC = 10.0

OUTPUT_DIR = Path("/kaggle/working/ref_audio_candidates")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 該当する発話の抽出
candidates = df[
    (df["speaker"] == TARGET_SPEAKER) &
    (df["duration"] >= MIN_SEC) &
    (df["duration"] <= MAX_SEC)
]

print(f"🎯 条件一致件数 ({TARGET_SPEAKER}, {MIN_SEC}s ~ {MAX_SEC}s): {len(candidates)} 件\n")

for idx, row in candidates.iterrows():
    start_s = row["start"]
    end_s = row["end"]
    dur = row["duration"]
    
    out_path = OUTPUT_DIR / f"{TARGET_SPEAKER}_clip_{idx}_{start_s}s.wav"
    
    cmd = [
        "ffmpeg", "-y",
        "-ss", str(start_s),
        "-to", str(end_s),
        "-i", AUDIO_FILE_PATH,
        "-ac", "1",
        "-ar", "16000",
        "-acodec", "pcm_s16le",
        str(out_path)
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"保存完了: {out_path.name} (長さ: {dur}秒 / 開始: {start_s}秒)")

print(f"\n✅ 抽出ファイルを保存しました: {OUTPUT_DIR}")